In [22]:
import copy
import random
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.modules.linear import Linear

 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42                                           

### Task 13 — Toolkit v2

Extend the functions from Task 7 to support dropout regularization and real-world datasets.

1. **`set_seed(seed)`**: Sets the random seed across `random`, `numpy`, and `tensorflow` to ensure experiment reproducibility.
2. **`build_model(input_dim, hidden_layers, activation, dropout_rate, output_units, output_activation)`**: Constructs an MLP model, adding a `Dropout(dropout_rate)` layer after each hidden layer only when `dropout_rate > 0`.
3. **`train_model(model, X_train, y_train, epochs, loss, metrics, patience)`**: Compiles and fits the model; when `patience` is provided, attaches `EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)`.
4. **`results_table(rows)`**: Converts a list of result dictionaries into a sorted pandas `DataFrame`.
5. **`plot_histories(histories, metric)`**: Plots the selected metric across multiple models on a single comparison chart.

In [17]:
def set_seed(seed=42):
    """Set seed for random, numpy and torch"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)             
    torch.cuda.manual_seed_all(seed)   


set_seed(SEED)

In [19]:
ACTIVATIONS = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid, "elu": nn.ELU}


def get_activation(name) -> None | Softmax | Any:
    """Maps an activation name to a PyTorch layer; returns None for None and 'linear'."""
    if name in (None, "linear"):
        return None
    if name == "softmax":
        return nn.Softmax(dim=1)
    return ACTIVATIONS[name]()

In [25]:
def make_hidden_block(in_features, units, activation, dropout_rate) -> list[Linear]:
    """Single hidden layer: Linear -> activation -> Dropout (only when dropout_rate > 0)."""
    block = [nn.Linear(in_features, units)]
    
    if get_activation(activation) is not None:
        block.append(get_activation(activation))
    
    if dropout_rate > 0:
        block.append(nn.Dropout(dropout_rate))
    
    return block

In [26]:
def make_output_block(in_features, output_units, output_activation) -> list[Linear]:

    block = [nn.Linear(in_features, output_units)]
    
    if get_activation(output_activation) is not None:
        block.append(get_activation(output_activation))
    
    return block

In [24]:
def build_model(input_dim, hidden_layers, activation="relu", dropout_rate=0.0,
                output_units=1, output_activation=None) -> Sequential:
    """Builds a network from hidden_layers and an output layer.

    hidden_layers=[] yields a linear model (output layer only).
    """
    layers = []
    in_features = input_dim
    
    for units in hidden_layers:
        layers += make_hidden_block(in_features, units, activation, dropout_rate)
        in_features = units                  
    
    layers += make_output_block(in_features, output_units, output_activation)
    
    return nn.Sequential(*layers).to(DEVICE)

In [23]:
def count_params(model):
    """Number of params in models."""
    return sum(p.numel() for p in model.parameters())

In [28]:
example = build_model(input_dim=8, hidden_layers=[64, 32], dropout_rate=0.2)
print(example)
print("Num of params", count_params(example))
print

Sequential(
  (0): Linear(in_features=8, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=64, out_features=32, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=32, out_features=1, bias=True)
)
Num of params 2689


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [30]:
# Same model in keras:
# model = keras.Sequential([
#    keras.Input(shape=(8,)),
#    layers.Dense(64, activation="relu"),
#    layers.Dropout(0.2),
#    layers.Dense(32, activation="relu"),
#    layers.Dropout(0.2),
#    layers.Dense(1),
# ])
# model.summary()   # Total params: 8*64 + 64*32+32 + 32*1+1 = 2,689

Unlike Keras, which encapsulates the entire pipeline within `fit()`, PyTorch breaks training down into explicit, modular steps:

1. **`to_tensors`**: Convert NumPy arrays to `float32` tensors; reshape target `y` from `(N,)` to `(N, 1)` to match the model's output dimensions.
2. **`split_validation`**: Allocate the final 20% of data for validation (identical to `validation_split` in Keras).
3. **`make_loader`**: Wrap training data into a `DataLoader` to shuffle and generate batches of 32 samples.
4. **`train_step`**: Execute a single optimization step: zero gradients → forward pass → compute loss → backward pass (`loss.backward()`) → update weights (`optimizer.step()`).
5. **`train_one_epoch`**: Run `train_step` across all training batches with dropout active (`model.train()`).
6. **`evaluate_loss`**: Compute validation loss with dropout disabled (`model.eval()`) and gradient calculation turned off (`torch.no_grad()`).
7. **`EarlyStopping`**: Track the best model state and halt execution when `val_loss` fails to improve for `patience` consecutive epochs.
8. **`train_model`**: Coordinate steps 1–7 and return the training history dictionary: `{'loss': [...], 'val_loss': [...], ...}`.

In [32]:
def to_tensors(X, y):
    """Converts NumPy arrays to float32 tensors; reshapes y from (N,) to (N, 1) to match model output.

    Without reshaping, MSELoss would compare (N, 1) with (N,), silently broadcasting
    them into an (N, N) matrix, which would corrupt model training.
    """
    X = torch.as_tensor(np.asarray(X), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(len(y), -1)
    return X, y


def split_validation(X, y, validation_split):
    """Splits off the LAST validation_split fraction of data as the validation set (identical to Keras validation_split)."""
    n_train = len(X) - int(len(X) * validation_split)
    return (X[:n_train], y[:n_train]), (X[n_train:], y[n_train:])


def make_loader(X, y, batch_size):
    """Splits data into shuffled batches of size batch_size."""
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

In [33]:
LOSSES = {
    "mse": nn.MSELoss,
    "mae": nn.L1Loss,
    "binary_crossentropy": nn.BCEWithLogitsLoss,
}


def get_loss_fn(name):
    """Keras-style loss name ('mse', 'mae', ...) -> PyTorch loss function."""
    return LOSSES[name]()


def compute_metric(name, output, y):
    """Computes the value of a single metric ('mae' or 'mse') for model output."""
    if name == "mae":
        return (output - y).abs().mean().item()
    if name == "mse":
        return ((output - y) ** 2).mean().item()
    raise ValueError(f"Unknown metric: {name}")


def compute_logs(loss_value, metrics, output, y):
    """Dictionary {'loss': ..., 'mae': ...}, mimicking Keras epoch output logs."""
    logs = {"loss": loss_value}
    for name in metrics:
        logs[name] = compute_metric(name, output, y)
    return logs

In [36]:
def train_step(model, xb, yb, loss_fn, optimizer):
    """Single training step on a single batch. Returns model output and loss value."""
    optimizer.zero_grad()  # 1. Clear gradients from previous step (otherwise they accumulate)
    output = model(xb)  # 2. Forward pass / predictions
    loss = loss_fn(output, yb)  # 3. Compute loss
    loss.backward()  # 4. Backpropagation: compute gradients w.r.t. weights
    optimizer.step()  # 5. Update weights
    return output.detach(), loss.item()


def train_one_epoch(model, loader, loss_fn, optimizer, metrics):
    """Single training epoch with dropout ENABLED. Returns average logs across all batches."""
    model.train()  # Training mode: dropout active
    sums, n_seen = {}, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        output, loss = train_step(model, xb, yb, loss_fn, optimizer)
        for key, value in compute_logs(loss, metrics, output, yb).items():
            sums[key] = (
                sums.get(key, 0.0) + value * len(xb)
            )  # Weighted by batch size
        n_seen += len(xb)
    return {key: total / n_seen for key, total in sums.items()}


def evaluate_loss(model, X, y, loss_fn, metrics):
    """Computes loss and metrics on a dataset with dropout DISABLED and autograd turned off."""
    model.eval()  # Evaluation mode: dropout disabled
    with torch.no_grad():
        X, y = X.to(DEVICE), y.to(DEVICE)
        output = model(X)
        return compute_logs(loss_fn(output, y).item(), metrics, output, y)

In [34]:
class EarlyStopping:
    """PyTorch equivalent of EarlyStopping(monitor='val_loss', patience=..., restore_best_weights=True)."""

    def __init__(self, patience):
        self.patience = patience
        self.best_loss = float("inf")
        self.best_weights = None
        self.bad_epochs = 0  # number of consecutive epochs without improvement

    def should_stop(self, val_loss, model):
        """Saves weights when val_loss improves; returns True when training should stop."""
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_weights = copy.deepcopy(model.state_dict())
            self.bad_epochs = 0
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore(self, model):
        """Loads the best-saved weights back into the model."""
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)

In [35]:
def append_history(history, logs, prefix=""):
    """Appends single-epoch logs to the history dict; prefix='val_' for validation."""
    for key, value in logs.items():
        history.setdefault(prefix + key, []).append(value)


def train_model(
    model,
    X_train,
    y_train,
    epochs=100,
    loss="mse",
    metrics=("mae",),
    patience=None,
    batch_size=32,
    validation_split=0.2,
    lr=1e-3,
):
    """Trains the model (equivalent to compile + fit). Returns a history dict like Keras history.history.

    patience=None: disables early stopping.
    patience=int: EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True).
    """
    loss_fn = get_loss_fn(loss)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr
    )  # Keras-style 'adam', lr=0.001

    X, y = to_tensors(X_train, y_train)
    (X_tr, y_tr), (X_val, y_val) = split_validation(X, y, validation_split)
    loader = make_loader(X_tr, y_tr, batch_size)
    stopper = EarlyStopping(patience) if patience is not None else None

    history = {}
    for _ in range(epochs):
        append_history(
            history, train_one_epoch(model, loader, loss_fn, optimizer, metrics)
        )
        append_history(
            history,
            evaluate_loss(model, X_val, y_val, loss_fn, metrics),
            prefix="val_",
        )
        if stopper and stopper.should_stop(history["val_loss"][-1], model):
            break

    if stopper:
        stopper.restore(model)  # restore_best_weights=True
    return history


def predict(model, X):
    """Returns single-output model predictions as a 1D NumPy array (supports PyTorch or scikit-learn models)."""
    if not isinstance(model, nn.Module):  # scikit-learn models use their own .predict()
        return model.predict(X)
    model.eval()  # disables dropout and batchnorm, matching Keras predict()
    with torch.no_grad():
        X = torch.as_tensor(np.asarray(X), dtype=torch.float32).to(DEVICE)
        return model(X).cpu().numpy().ravel()

In [37]:
def results_table(rows, sort_by="MAE", ascending=True):
    """Converts a list of results (one dictionary per model) into a table sorted by sort_by."""
    table = pd.DataFrame(rows).sort_values(sort_by, ascending=ascending)
    return table.reset_index(drop=True).round(4)

In [38]:
def plot_histories(histories, metric="loss", show_train=True, show_val=True,
                   logy=False, title=None, ax=None):
    """Plots a metric across multiple model histories on a single plot.
    
    histories: {model_name: history from train_model}.
    Color = model, solid line = train, dashed line = validation.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))
    for i, (name, history) in enumerate(histories.items()):
        color = f"C{i % 10}"                    
        if show_train:
            ax.plot(history[metric], color=color, label=f"{name}: {metric}")
        if show_val:
            ax.plot(history[f"val_{metric}"], color=color, linestyle="--",
                    label=f"{name}: val_{metric}")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("epoka")
    ax.set_ylabel(metric)
    ax.set_title(title or metric)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    return ax